# Example 10.4.3 / 4: Bound the Argument Domain

Companion notebook for **10.4.3 Automatic Piecewise-Linear Approximation** in the AMPL lab Documentation, section "Bound the Argument Domain". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Bound the Argument Domain

A PWL approximation needs a finite interval. MP builds it from the argument's bounds, including
bounds it deduces from the constraints, and then applies option `cvt:plapprox:domain` (synonyms
`plapprox:domain`, `plapproxdomain`): "both arguments and result are bounded to
±[pladomain]", default `1e6`. Those bounds are **hard limits** that the original model did not
contain. When a variable has no bounds of its own, the default can cut off the true optimum
without any infeasibility:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param budget > 0;  # upper limit on exp(Growth)
var Growth;                       # no bounds

maximize Level: Growth;  # push Growth as high as the budget allows
subject to BudgetLimit: exp(Growth) <= budget;  # nonlinear, approximated by MP

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["budget"] = 1e8

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result)
print(ampl.var["Growth"].value(), ampl.get_value("exp(Growth)"))

solved
13.81551055796427 999999.9999999959


### Pure AMPL: data.dat and model.run

Pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param budget := 1e8;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
solve;
display Growth, exp(Growth);

HiGHS 1.14.0: 
------------ WARNINGS ------------
  Argument domain of a 'Exp'
has been reduced to [-230.258500, 13.815511] for numerical reasons
(partially controlled by cvt:plapprox:domain.)
  An expression of type 'Exp' has been
piecewise-linearly approximated. Set cvt:plapprox:reltol
to control precision (currently 0.010000).
Growth = 13.8155
exp(Growth) = 1e+06



### Another way in amplpy

The exact optimum is $\log(10^8) \approx 18.42$, but the solve stops at $\log(10^6) \approx 13.82$
with status `solved`: the `PLApproxDomain_Exp` warning shows that the argument domain was reduced
so that `exp(Growth)` stays within `1e6`.

> **Caution**
> The domain bound changes the feasible set silently. Read every `PLApproxDomain_...` warning,
> and give each approximated argument and result explicit, meaningful bounds. A result that sits
> exactly at a domain limit is a symptom, not an optimum.

**Raise the domain.** When no exact rewrite exists, raise the domain only as far as the model
needs. With the same model and data, `cvt:plapprox:domain=1e8` admits the true optimum, but the
approximation's coefficients now span a much wider range:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.param["budget"] = 1e8

In [11]:
# run.py
ampl.solve(solver="highs", verbose=False, highs_options="cvt:plapprox:domain=1e8")
print(ampl.solve_result)
print(ampl.var["Growth"].value())

solved
18.42068073250419


### Pure AMPL: data.dat and model.run

Pure AMPL, with the same `model.mod`:

In [12]:
%%writefile data_2.dat
param budget := 1e8;

Writing data_2.dat


In [13]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data_2.dat;

In [14]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
option highs_options 'cvt:plapprox:domain=1e8';
solve;
display Growth;

HiGHS 1.14.0:   cvt:plapprox:domain = 100000000
  Warning code 1 for call loader().Highs_addRows(lp(), acc_constraints_.lb.size(), acc_constraints_.lb.data(), acc_constraints_.ub.data(), acc_constraints_.coeffs.size(), acc_constraints_.starts.data(), acc_constraints_.indices.data(), acc_constraints_.coeffs.data())

------------ WARNINGS ------------
  Argument domain of a 'Exp'
has been reduced to [-230.258500, 18.420681] for numerical reasons
(partially controlled by cvt:plapprox:domain.)
  An expression of type 'Exp' has been
piecewise-linearly approximated. Set cvt:plapprox:reltol
to control precision (currently 0.010000).
Growth = 18.4207

